<a href="https://colab.research.google.com/github/meltyblend/NeuralNet_from_scratch/blob/main/Fashion_MNIST_classifier_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"></a>

# Construindo uma Melhor Compreensão de Classificadores de Imagens

Vamos trabalhar com o [conjunto de dados Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist). Os arquivos CSV vieram do [Kaggle](https://www.kaggle.com/datasets/zalando-research/fashionmnist).

O conjunto de dados Fashion-MNIST se mostra um pouco mais desafiador do que o [conjunto MNIST original](http://yann.lecun.com/exdb/mnist/) de números desenhados à mão em imagens 28x28.

Eu me inspirei bastante em [Samson Zhang](https://www.youtube.com/watch?v=w8yWXqWQYmU). Eles realmente despertaram meu desejo de aprender o máximo possível sobre redes neurais [link para o vídeo deles](https://www.youtube.com/watch?v=w8yWXqWQYmU).




In [1]:
import numpy as np
import pandas as pd
import math
from matplotlib import pyplot as plt

import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")

Using Colab cache for faster access to the 'fashionmnist' dataset.


# Propagação Direta

Informações sobre nosso conjunto de dados: estamos trabalhando com um conjunto de imagens em que cada imagem tem 784 pixels.

Nossa rede neural será muito simples, utilizando apenas 3 camadas: *Entrada, Oculta e Saída*.

## Camada de Entrada

Nossa Camada de Entrada é muito simples. Ela é a própria imagem!

Nossa imagem tem 784 pixels, e cada pixel contém um valor de 0 a 255 (0 = preto, 255 = branco).

Vamos alimentar essa imagem na nossa rede neural por meio de um método conhecido como *Propagação Direta*.

## Camada Oculta

As Camadas Ocultas são um pouco misteriosas. Entender quantos neurônios você precisa pode ser difícil. Adicione neurônios demais e você pode causar *overfitting* ou o tempo de treinamento pode ficar extremamente longo.

Por enquanto, para manter as coisas simples, vamos usar apenas 10 neurônios, para que possamos focar nas partes importantes e **divertidas** dos fundamentos de uma rede neural!

## Camada de Saída

É aqui que nossa rede neural chegará a algum tipo de conclusão com base nas probabilidades geradas pelos cálculos anteriores!

Nossa camada de saída tem 10 *classificações* baseadas nas classes fornecidas pelo conjunto de dados Fashion-MNIST.

Elas são: `{Camiseta/Top, Calça, Suéter, Vestido, Casaco, Sandália, Camisa, Tênis, Bolsa, Bota de cano curto}`

# Leitura do Conjunto de Dados

Vamos começar lendo e testando nosso conjunto de dados

In [2]:
data = pd.read_csv(path+"/fashion-mnist_train.csv")

## Dando uma Olhada no Nosso Conjunto de Dados

In [3]:
data.head()

In [4]:
# isto mostrará as dimensões do nosso conjunto de dados
data.shape

Podemos ver que temos 60.000 linhas e 785 colunas.

Isso é bom, mas mais tarde precisaremos transpor nossos dados para que possamos trabalhar com nossas imagens como colunas.

Se trabalhássemos com o conjunto de dados sem transpor:
1. Isso simplesmente estaria muito incorreto
2. Como nosso conjunto de dados tem 60.000, isso significa que temos 60.000 variáveis correspondentes a imagens e não aos dados reais.

Então precisamos transpor para trabalhar com os pixels individuais e suas variáveis associadas!

# Codificação na Prática!

Vamos começar vetorizando nosso conjunto de dados para que não trabalhemos diretamente com o arquivo CSV

In [5]:
data_vec = np.array(data) # isso transforma tudo no nosso csv em um array

# definindo linhas (m) e colunas (n)
m, n = data_vec.shape
# m = 60.000 n = 785

# embaralhando nossos dados como um baralho de cartas
# fazemos isso porque queremos ter certeza de que o treinamento será
# em um conjunto aleatório de imagens a cada execução
np.random.shuffle(data_vec)

## Dividindo Nosso Conjunto de Treinamento

Queremos separar nosso conjunto de treinamento. Uma parte será usada para treinar nossa rede neural, e a outra parte será usada para testar.

Apesar de nosso conjunto de dados fornecer um conjunto de teste separado, não queremos usá-lo até o final.

Pense nesse conjunto de teste final como o exame final da nossa rede neural, e no nosso conjunto de teste criado a partir da divisão do treinamento como uma prova intermediária!

In [6]:
# transpondo as primeiras 1000 linhas do arquivo de treinamento
data_dev = data_vec[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n] / 255. # normaliza os valores dos pixels para ficarem entre 0 e 1

# transpondo o restante do conjunto de dados para fins reais de treinamento
# neste ponto temos um conjunto de treinamento com 59.000 imagens
# estes são os dados que utilizaremos para atualizar pesos e vieses
# na nossa rede neural
data_train = data_vec[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255. # normaliza os valores dos pixels

# Hora da Matemática!

Agora que já tiramos a maior parte da parte preliminar do caminho, podemos mergulhar de vez!

Começando com **Propagação Direta**

Como dissemos antes, nossa rede neural tem 3 camadas. Agora vamos explicar como cada camada interage matematicamente com as outras!

## Transpondo Nossos Dados
Considere $A^{[0]} = X$

Onde,
$X =
\begin{bmatrix}
\cdots & x_1 & \cdots \\n\cdots & x_2 & \cdots \\n & \vdots & \\n \cdots & x_k & \cdots \\n\end{bmatrix}$

Esta matriz representa nosso conjunto de dados original, onde ${x_1, x_2, \dots, x_n}$ representa nossas 60.000 linhas de imagens, mas, como dissemos anteriormente, queremos que isso seja transposto, então vamos transpor!

Considere $A^{[0]} = X^T$

$X^T =
\begin{bmatrix}
\vdots & \vdots & & \vdots \\n x_1 & x_2 & \cdots & x_k \\n\vdots & \vdots & & \vdots \\n\end{bmatrix}$

Muito melhor agora!

Isso representa onde estamos atualmente no código.

## Passando Nossa Matriz Pela Rede Neural

Para alimentar nossas imagens através da rede neural, precisamos definir um *neurônio*!

**Neurônios** são uma soma ponderada com algum viés.

$$Z^{[l]} = W^{[l]}X^{[l-1]}+b^{[l]}$$

Como definimos $A = X$, nossa fórmula se torna

$$Z^{[l]} = W^{[l]}A^{[l-1]}+b^{[l]}$$

Agora isso estranhamente parece exatamente com outra fórmula que todos nós deveríamos conhecer e amar. Essa é a fórmula de uma reta no plano: $y = mx+b$. É um tanto trabalhoso perceber o quão simples algumas dessas ideias são...

No entanto, a intuição por trás delas é extremamente difícil de compreender.

### Detalhando Nossa Fórmula

$Z^{[n]} = $ Saída da Camada Oculta

$W^{[n]} = $ Pesos

$b^{[n]} = $ Viés

No início, nossos *Pesos* e *Vieses* serão escolhidos aleatoriamente. Vamos detalhar mais adiante como treinamos esses pesos e vieses para obter previsões mais precisas.

## Encontrando $Z^{[1]}$ — Saída da Camada Oculta

$Z^{[1]}$ corresponde à saída da nossa primeira *Camada Oculta*

$$Z^{[1]} = W^{[1]}A^{[0]} + b^{[1]}$$

Vamos analisar as dimensões de nossas matrizes e vetores na fórmula

$$Z^{[1]} = dim(W^{[1]})dim(A^{[0]}) + dim(b^{[1]})$$

$$Z^{[1]} = (10 \times 784)(784 \times k) + (10 \times 1)$$

Onde $k$ é o número de imagens sendo passadas pela rede neural.

$$= (10 \times k) + (10 \times 1)$$

$$Z^{[1]} = (10 \times k)$$

No final, ficamos com a saída da matriz da nossa *Camada Oculta*.

## Funções de Ativação

As *Funções de Ativação* são o que torna a matemática *não linear*, o que significa que vamos aplicar uma função à saída $Z^{[1]}$ para transformar nossa função linear em algo curvo, para que mais adiante...

Há algumas funções de ativação, porém vamos trabalhar com *ReLU* (Rectified Linear Unit).

$$ReLU(x) = \max(0,x)=
\begin{cases}
 x, & \text{se } x > 0 \\n 0, & \text{se } x \leq 0
\end{cases}
$$

Em essência, estamos usando ReLU para criar um vetor de números não negativos.

Agora vamos aplicar $ReLU$,

$A^{[1]} = ReLU(Z^{[1]})$

$$A^{[1]} =
\begin{bmatrix}
\max(0,z_{11}^{[1]}) & \max(0,z_{12}^{[1]}) & \dots & \max(0,z_{1j}^{[1]}) \\n\max(0,z_{21}^{[1]}) & \max(0,z_{22}^{[1]}) & \dots & \max(0,z_{2j}^{[1]}) \\n\vdots & \vdots & \ddots & \vdots \\n\max(0,z_{i1}^{[1]}) & \max(0,z_{i2}^{[1]}) & \dots & \max(0,z_{ij}^{[1]}) \\n\end{bmatrix}
$$

$dim(Z^{[1]}) = dim(A^{[1]})$

Assim, a saída da nossa *Camada Oculta* se torna não linear e pronta para ser usada na próxima camada

### Gráfico da ReLU
![image.png](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAC9AAAAZkCAYAAAC3DgxpAAAMTmlDQ1BJQ0MgUHJvZmlsZQAASImVVwdYU8kWnltSSQgQiICU0JsgIiWAlBBaAOlFEJWQBAglxoSgYkcXFVy7iGBFV0EUXV0BWWyoq64sit21[...]

## Encontrando $Z^{[2]}$ — Saída da Camada de Saída

Precisamos repetir o processo de forma semelhante ao que fizemos para encontrar $Z^{[1]}$

$$Z^{[2]} =  W^{[2]}A^{[1]} + b^{[2]}$$

$$ = (10 \times 10) (10 \times k) + (10 \times 1)$$

$$= (10 \times k)$$

Assim, obtemos a saída da nossa Camada de Saída.

No entanto, se olharmos mais de perto nossa saída (por enquanto vamos assumir que $k = 1$, o que significa que estamos vendo uma única imagem)

$dim(Z^{[2]}) = (10 \times 1)$

$$Z^{[2]} =
\begin{bmatrix}
 z_1 \\n z_2 \\n \vdots \\n z_k
\end{bmatrix}
$$

Cada elemento dentro desse vetor agora nos dirá uma probabilidade de como a rede neural acredita que a imagem pode ser classificada.

No entanto, os resultados não são exatamente o que estamos procurando, então precisamos normalizar o vetor introduzindo a função *Softmax*

## Função Softmax

A função *Softmax* nos ajuda a interpretar melhor as probabilidades de qual classificação a nossa rede neural acredita que a imagem de entrada possa representar.

$$Softmax(Z^{[2]}) = \hat{y}_i = \dfrac{e^{Z_i^{[2]}}}{\sum_{j=1}^{10} e^{Z_j^{[2]}}}$$

Quem prestar bastante atenção pode notar que essa função normaliza nossos dados para que a soma de cada elemento seja igual a 1.

Isso nos ajuda a interpretar melhor a probabilidade de a imagem ser de `Calças` ou de uma `Camiseta`

Estamos elevando cada elemento em $Z^{[2]}$ à potência de $e$ e depois dividindo pelo somatório das exponenciais de cada elemento!

$$Softmax(Z^{[2]}) = \dfrac{1}{e^{z_1} + e^{z_2} + \dots + e^{z_{10}}}\n\begin{bmatrix}\n e^{z_1} \\n e^{z_2} \\n \vdots \\n e^{z_{10}}\n\end{bmatrix}\n$$

Se você observar com ainda mais atenção, deve ter notado que estamos dividindo pela **norma Manhattan** ou pela **norma $L_1$**

# Codificando a Propagação Direta!!! Finalmente!!!

## Inicializando Parâmetros

Primeiro vamos criar uma função que inicializa nossos *Pesos* e *Vieses*

In [7]:
def init_params():
  # para a primeira camada (Camada Oculta)
  W1 = np.random.rand(10,784) - 0.5
  b1 = np.random.rand(10,1) - 0.5

  # para a segunda camada (Camada de Saída)
  W2 = np.random.rand(10,10) - 0.5
  b2 = np.random.rand(10,1) - 0.5
  return W1, b1, W2, b2

No início, queremos que nossos *Pesos* e *Vieses* tenham alguns valores. Então escolhemos valores aleatórios entre $[0,1]$.

**Exemplo** do que a primeira linha da função está fazendo

`W1` tem 10 linhas e 784 colunas, em que cada elemento é um valor aleatório entre 0 e 1

Como estamos subtraindo 0,5, estamos centralizando cada elemento em torno de 0

Se nosso menor elemento for 0, então $0 - 0.5 = -0.5$

Se nosso maior elemento for 1, então $1 - 0.5 = 0.5$

Portanto, nosso novo intervalo para todos os números passa a ser $[-0.5, 0.5]$

Queremos isso porque, quando aplicarmos nossa *Função de Ativação*, podemos ignorar os valores negativos; os valores negativos indicam que a rede neural acredita que eles não fazem fortemente parte da...

# Funções de Ativação

Antes de começarmos a nossa *Propagação Direta*, precisamos criar nossas *Funções de Ativação*

In [8]:
# isto é exatamente o que definimos antes
# comparamos cada elemento com 0
# x, x > 0
# 0, x <=0


#def relu(Z):
#  return np.maximum(0,Z)

#### Testando com GELU

Recentemente aprendi sobre a Unidade Linear de Erro Gaussiano (Gaussian Error Linear Unit) e, pelo que entendemos, o GELU atua como uma função de ativação melhor em comparação com o ReLU. Isso vem principalmente da ideia de que...

$$GELU(x) =0.5 \cdot x \cdot \left( 1 + \tanh\left( \sqrt{ \dfrac{2}{\pi} } \cdot (x + 0.44715) * x^3  \right) \right) $$

In [9]:
def gelu(Z):
  return 0.5 * Z * (1 + np.tanh(np.sqrt(2 / np.pi) * (Z + 0.044715 * Z ** 3)))

# Função Softmax

Esta é a nossa função softmax. Mais uma vez, isso fará com que cada valor fique entre $[0,1]$, onde a soma de todos os elementos após a aplicação da softmax é igual a $1$